# QZero: Synthetic Data & QSVM Demo

This notebook demonstrates the end-to-end pipeline for the QZero project using synthetic data that simulates the UGRansome dataset.

### Pipeline Steps:
1. **Data Generation**: Create synthetic network traffic data with normal, anomaly (zero-day), and signature traffic.
2. **Preprocessing**: Scale and split the data.
3. **Feature Engineering**: Reduce dimensions and scale for quantum encoding.
4. **QSVM Training**: Train a Quantum Support Vector Machine (or classical fallback) on the processed data.
5. **Evaluation**: Measure accuracy and zero-day detection capabilities.

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing.data_loader import generate_synthetic_data, DataLoader
from src.preprocessing.feature_engineering import FeatureEngineer
from src.quantum.qsvm import QSVM, QSVMConfig, compare_quantum_classical

%matplotlib inline
sns.set_style("whitegrid")

## 1. Generate Synthetic Data

We generate 1000 samples with 14 features (mimicking UGRansome attributes).
- **Normal**: Standard traffic
- **Anomaly**: Zero-day attacks (unknown patterns)
- **Signature**: Known attacks (specific patterns)

In [ ]:
X_raw, y_raw = generate_synthetic_data(
    n_samples=1000,
    n_features=14,
    anomaly_ratio=0.15,
    signature_ratio=0.20,
    random_state=42,
    difficulty="medium"
)

print(f"Data shape: {X_raw.shape}")
print("Class distribution:")
print(y_raw.value_counts().rename({0: 'Normal', 1: 'Anomaly (Zero-day)', 2: 'Signature'}))

In [ ]:
# Visualize first two features
plt.figure(figsize=(10, 6))
sns.scatterplot(x=X_raw.iloc[:, 0], y=X_raw.iloc[:, 1], hue=y_raw, palette="viridis", alpha=0.6)
plt.title("Synthetic Traffic Data (First 2 Features)")
plt.xlabel("Flow Duration")
plt.ylabel("Total Fwd Packets")
plt.legend(title="Class")
plt.show()

## 2. Preprocessing & Splitting

We convert the problem to binary classification: **Normal (0)** vs **Attack (1)**.
Then split into Train/Val/Test.

In [ ]:
loader = DataLoader()
data = loader.prepare_data(X_raw, y_raw, binary=True, test_size=0.2, val_size=0.1)

print(f"Train set: {data['X_train'].shape}")
print(f"Test set:  {data['X_test'].shape}")

## 3. Quantum Feature Engineering

Quantum circuits on NISQ devices are limited in width (qubits). We reduce the 14 features down to 4 using PCA and scale them to $[0, \pi]$ for angle encoding.

In [ ]:
N_QUBITS = 4

engineer = FeatureEngineer(n_qubits=N_QUBITS, reduction_method="pca", encoding_range=(0, np.pi))

# Fit on training data, transform both
X_train_q = engineer.fit_transform(data["X_train"], data["y_train"])
X_test_q = engineer.transform(data["X_test"])

print(f"Quantum Features shape: {X_train_q.shape}")
print(f"Range: [{X_train_q.min():.2f}, {X_train_q.max():.2f}]")
print(f"Explained variance ratio: {engineer.get_feature_importance()}")

## 4. QSVM Training

We train a Quantum Support Vector Machine using a Fidelity Quantum Kernel.
- **Feature Map**: ZZFeatureMap (entangled)
- **Kernel**: Fidelity between quantum states
- **Backend**: Local simulator (Statevector)

In [ ]:
# Configure QSVM
config = QSVMConfig(
    n_qubits=N_QUBITS,
    feature_map_type="zz",
    max_samples_quantum=200,  # Limit samples for speed in demo
    use_classical_fallback=True  # Use RBF if Qiskit not installed
)

qsvm = QSVM(config)

print("Training QSVM...")
result = qsvm.fit_evaluate(X_train_q, data["y_train"], X_test_q, data["y_test"])

print("\n--- Results ---")
print(f"Used Quantum Kernel: {result.used_quantum}")
print(f"Accuracy: {result.accuracy:.4f}")
print(f"F1 Score: {result.f1_score:.4f}")
print(f"Training Time: {result.train_time:.4f}s")
print(f"Support Vectors: {result.n_support_vectors}")

## 5. Classical vs Quantum Comparison

Comparing the QSVM against a classical SVM with RBF kernel on the same reduced features.

In [ ]:
q_res, c_res = compare_quantum_classical(X_train_q, data["y_train"], X_test_q, data["y_test"], n_qubits=4)

results_df = pd.DataFrame({
    "Method": ["QSVM (Quantum)", "SVM (Classical)"],
    "Accuracy": [q_res.accuracy if q_res else 0, c_res.accuracy],
    "F1 Score": [q_res.f1_score if q_res else 0, c_res.f1_score],
    "Time (s)": [q_res.train_time if q_res else 0, c_res.train_time]
})

print(results_df)